Generate Sample Dataset

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

df = pd.DataFrame({
    'month': np.random.choice(['Jan','Feb','Mar','Apr','May','Jun'], n),
    'region': np.random.choice(['North','South','East','West'], n),
    'product': np.random.choice(['Laptop','Phone','Tablet','Watch','Headphones'], n),
    'age_group': np.random.choice(['18-25','26-35','36-45','46-55','55+'], n),
    'sales': np.random.randint(100, 5000, n),
    'units': np.random.randint(1, 50, n)
})

df.to_csv('sales_data.csv', index=False)
print(f"Dataset created: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head())

Dataset created: 200 rows, 6 columns
  month region product age_group  sales  units
0   Apr   West  Tablet     46-55   2148     23
1   May   West   Watch     26-35   4246     10
2   Mar   West  Tablet     18-25   4880     44
3   May  North  Laptop       55+    816      2
4   May   West  Laptop     46-55   4644     13


Import Libraries

In [5]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()
print("Imports Successful!!!")

Imports Successful!!!


Initialize Claude

In [6]:
llm = ChatAnthropic (
    model = "claude-sonnet-4-6",
    temperature = 0
)
print("Claude ready!!!")

Claude ready!!!


Load Dataset

In [17]:
df = pd.read_csv("sales_data.csv")
print(f"Number of rows:  {df.shape[0]} rows, {df.shape[1]} columns")
print (df.head())

Number of rows:  200 rows, 6 columns
  month region product age_group  sales  units
0   Apr   West  Tablet     46-55   2148     23
1   May   West   Watch     26-35   4246     10
2   Mar   West  Tablet     18-25   4880     44
3   May  North  Laptop       55+    816      2
4   May   West  Laptop     46-55   4644     13


Define Agent tools

In [18]:
@tool
def analyze_data(question: str) -> str:
    """Analyzes the sales dataset and answers questions about it.
    Use for questions about counts, averages, totals, and data exploration."""
    try:
        result = []
        result.append(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
        result.append(f"Columns: {list(df.columns)}")
        result.append(f"\nBasic Statistics:\n{df.describe().to_string()}")
        result.append(f"\nSales by Region:\n{df.groupby('region')['sales'].sum().to_string()}")
        result.append(f"\nSales by Product:\n{df.groupby('product')['sales'].sum().to_string()}")
        result.append(f"\nSales by Month:\n{df.groupby('month')['sales'].sum().to_string()}")
        return "\n".join(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def plot_bar_chart(column: str) -> str:
    """Creates a bar chart for sales by a specific column.
    Valid columns: region, product, month, age_group"""
    try:
        plt.figure(figsize=(10, 6))
        data = df.groupby(column)['sales'].sum().sort_values(ascending=False)
        sns.barplot(x=data.index, y=data.values, palette='viridis')
        plt.title(f'Total Sales by {column.title()}', fontsize=14)
        plt.xlabel(column.title())
        plt.ylabel('Total Sales ($)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(f'bar_chart_{column}.png')
        plt.show()
        return f"Bar chart created for sales by {column}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def plot_pie_chart(column: str) -> str:
    """Creates a pie chart showing sales distribution by a specific column.
    Valid columns: region, product, month, age_group"""
    try:
        plt.figure(figsize=(8, 8))
        data = df.groupby(column)['sales'].sum()
        plt.pie(data.values, labels=data.index, autopct='%1.1f%%', 
                colors=sns.color_palette('viridis', len(data)))
        plt.title(f'Sales Distribution by {column.title()}', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'pie_chart_{column}.png')
        plt.show()
        return f"Pie chart created for sales distribution by {column}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def plot_scatter(x_col: str, y_col: str) -> str:
    """Creates a scatter plot to show relationship between two numeric columns.
    Valid columns: sales, units"""
    try:
        plt.figure(figsize=(10, 6))
        sns.scatterplot(data=df, x=x_col, y=y_col, 
                       hue='region', palette='viridis', alpha=0.7)
        plt.title(f'{x_col.title()} vs {y_col.title()}', fontsize=14)
        plt.xlabel(x_col.title())
        plt.ylabel(y_col.title())
        plt.tight_layout()
        plt.savefig(f'scatter_{x_col}_{y_col}.png')
        plt.show()
        return f"Scatter plot created for {x_col} vs {y_col}"
    except Exception as e:
        return f"Error: {str(e)}"

tools = [analyze_data, plot_bar_chart, plot_pie_chart, plot_scatter]
llm_with_tools = llm.bind_tools(tools)
print(f"Tools ready: {[t.name for t in tools]}")

Tools ready: ['analyze_data', 'plot_bar_chart', 'plot_pie_chart', 'plot_scatter']


Conversation Memory

In [21]:
class ConversationMemory:
    def __init__(self):
        self.messages = [
            SystemMessage(content="""You are a data analysis assistant with visualization capabilities.
            You have access to tools to analyze data, plot bar chart, plot pie chart, plot scatter plots.
            Always use the appropriate tool to respond for the data analysis request accurately.
            Remember the context of our conversation and refer back to previous results when the user
            ask follow up questions.""")
        ]
    
    def add_human_message(self, content: str):
        self.messages.append(HumanMessage(content=content))

    def add_ai_message(self, message):
        self.messages.append(message)

    def add_tool_result(self, content: str, tool_call_id: str):
        self.messages.append(
            ToolMessage(content=content, tool_call_id=tool_call_id)
        )

    def get_messages(self):
        return self.messages

    def show_history(self):
        print("\n Conversation History:")
        print("-" * 54)
        for msg in self.messages:
            if isinstance(msg, SystemMessage):
                print(f" System: {msg.content[:54]}...")
            elif isinstance(msg, HumanMessage):
                print(f" You: {msg.content}")
            elif isinstance(msg, AIMessage):
                print(f" Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f" Tool Result: {msg.content}")
        print("-" * 54)

memory = ConversationMemory()
print("Conversation memory ready!!!")

Conversation memory ready!!!


Agent Loop

In [25]:
def run_conversational_agent():
    memory = ConversationMemory()
    tool_map = {t.name: t for t in tools}

    print(" Data Analysis Assistant with Visualization Capabilities Ready!!!")
    print(" You can ask to analyze the sales data and create charts like bar chart, pie chart, scatter plots!!!")
    print(" Type 'history' to see out conversation so far.")
    print(" Type 'quit' to exit.")
    print("=" * 54)

    while True:
        # Get user input
        user_input = input("\n You: ").strip()

        # Handle special commands
        if user_input.lower() == 'quit':
            print("\n Goodbye! Great Data Analysis Session!!!")
            break

        if user_input.lower() == 'history':
            memory.show_history()
            continue

        if not user_input:
            print("Please enter a question.")
            continue

        # Add user message to memory
        memory.add_human_message(user_input)

        #Get response from Claude
        response = llm_with_tools.invoke(memory.get_messages())
        memory.add_ai_message(response)

        # Process tool calls if any
        while response.tool_calls:
            for tool_call in response.tool_calls:
                print(f"\n Using tool: {tool_call['name']}")
                print(f"   Input: {tool_call['args']}")

                # Execute the tool
                selected_tool = tool_map[tool_call["name"]]
                tool_result = selected_tool.invoke(tool_call["args"])
                print(f"   Result: {tool_result}")

                # Save tool result to memory
                memory.add_tool_result(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            # Get final response after tool execution
            response = llm_with_tools.invoke(memory.get_messages())
            memory.add_ai_message(response)

        print(f"    Agent: {response.content}")
print("Agent loop ready!!!")

Agent loop ready!!!


Run the agent

In [ ]:
run_conversational_agent()